# Common preprocessing for diabetic dataset

Notebook này tạo một bản dữ liệu sạch chung sau EDA. File output sẽ được dùng làm đầu vào cho 3 hướng tiếp theo: phân loại, phân cụm và luật kết hợp.

Mục tiêu của bước chung:

- Đọc dữ liệu gốc `diabetic_data.csv`.
- Chuẩn hóa missing value từ `?` thành `NaN`.
- Loại bỏ các cột định danh hoặc gần như không có thông tin mô hình hóa.
- Xử lý một số giá trị bất thường/còn thiếu ở mức nền tảng.
- Tạo các feature dùng chung như `readmitted_binary`, nhóm tuổi dạng số, nhóm ICD-9.
- Lưu dữ liệu sạch ra `data/diabetic_data_clean_common.csv`.


In [ ]:
from pathlib import Path

try:
    import numpy as np
    import pandas as pd
except ImportError as exc:
    raise ImportError(
        "Notebook này cần pandas và numpy. Hãy cài bằng: pip install pandas numpy"
    ) from exc

pd.set_option("display.max_columns", 100)


In [ ]:
cwd = Path.cwd().resolve()
ROOT_DIR = cwd if (cwd / "diabetic_data.csv").exists() else cwd.parent
RAW_DATA_PATH = ROOT_DIR / "diabetic_data.csv"
OUTPUT_DIR = ROOT_DIR / "data"
OUTPUT_PATH = OUTPUT_DIR / "diabetic_data_clean_common.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_DATA_PATH, OUTPUT_PATH


## 1. Load dữ liệu gốc


In [ ]:
df_raw = pd.read_csv(RAW_DATA_PATH)
df = df_raw.copy()

print("Raw shape:", df.shape)
df.head()


## 2. Chuẩn hóa missing value

Trong dataset này, nhiều giá trị thiếu được ghi bằng dấu `?`, nên cần đổi sang `NaN` để xử lý nhất quán.


In [ ]:
df = df.replace("?", np.nan)

missing_summary = (
    df.isna()
    .sum()
    .to_frame("missing_count")
    .assign(missing_rate=lambda x: x["missing_count"] / len(df))
    .query("missing_count > 0")
    .sort_values("missing_rate", ascending=False)
)

missing_summary


## 3. Bỏ các cột không phù hợp cho bản clean chung

- `encounter_id`, `patient_nbr`: cột định danh, không đưa trực tiếp vào feature.
- `weight`: missing quá cao.
- `examide`, `citoglipton`: thường chỉ có một giá trị, gần như không mang thông tin phân biệt.


In [ ]:
columns_to_drop = [
    "encounter_id",
    "patient_nbr",
    "weight",
    "examide",
    "citoglipton",
]

df = df.drop(columns=[col for col in columns_to_drop if col in df.columns])

print("Shape after dropping common unused columns:", df.shape)
df.head()


## 4. Xử lý missing và giá trị bất thường mức nền tảng

Ở bước chung, ta chỉ xử lý các giá trị rõ ràng. Các chiến lược encode/scale chi tiết sẽ để riêng cho từng bài toán.


In [ ]:
unknown_fill_columns = ["race", "payer_code", "medical_specialty"]

for col in unknown_fill_columns:
    if col in df.columns:
        df[col] = df[col].fillna("Unknown")

if "gender" in df.columns:
    df = df[df["gender"] != "Unknown/Invalid"].copy()

print("Shape after basic missing handling:", df.shape)
df[unknown_fill_columns + ["gender"]].head()


## 5. Tạo biến mục tiêu nhị phân

Biến này phục vụ bài toán phân loại nhị phân. Bản clean chung vẫn giữ `readmitted` gốc để nếu cần có thể làm phân loại 3 lớp hoặc dùng trong phân tích luật kết hợp.


In [ ]:
if "readmitted" in df.columns:
    df["readmitted_binary"] = np.where(df["readmitted"].eq("NO"), 0, 1)

df[["readmitted", "readmitted_binary"]].head()


## 6. Tạo biến tuổi dạng số/thứ tự

`age` là nhóm tuổi có thứ tự. Bản clean chung giữ `age` gốc và thêm `age_midpoint`, `age_ordinal` để các bước sau chọn cách dùng phù hợp.


In [ ]:
age_order = [
    "[0-10)", "[10-20)", "[20-30)", "[30-40)", "[40-50)",
    "[50-60)", "[60-70)", "[70-80)", "[80-90)", "[90-100)",
]
age_midpoint_map = {
    "[0-10)": 5,
    "[10-20)": 15,
    "[20-30)": 25,
    "[30-40)": 35,
    "[40-50)": 45,
    "[50-60)": 55,
    "[60-70)": 65,
    "[70-80)": 75,
    "[80-90)": 85,
    "[90-100)": 95,
}
age_ordinal_map = {age: idx for idx, age in enumerate(age_order)}

if "age" in df.columns:
    df["age_midpoint"] = df["age"].map(age_midpoint_map)
    df["age_ordinal"] = df["age"].map(age_ordinal_map)

df[["age", "age_midpoint", "age_ordinal"]].head()


## 7. Gom nhóm mã chẩn đoán ICD-9

Các cột `diag_1`, `diag_2`, `diag_3` có rất nhiều mã khác nhau. Bản clean chung thêm các cột nhóm bệnh lớn để giảm số lượng giá trị phân loại.


In [ ]:
def map_icd9_group(code):
    if pd.isna(code):
        return "Unknown"

    code_str = str(code).strip()
    if code_str.startswith("V"):
        return "Supplementary_V"
    if code_str.startswith("E"):
        return "Supplementary_E"

    try:
        code_num = float(code_str)
    except ValueError:
        return "Other"

    if 1 <= code_num <= 139:
        return "Infectious_Parasitic"
    if 140 <= code_num <= 239:
        return "Neoplasms"
    if 240 <= code_num <= 279:
        return "Endocrine_Metabolic"
    if 280 <= code_num <= 289:
        return "Blood"
    if 290 <= code_num <= 319:
        return "Mental_Disorders"
    if 320 <= code_num <= 389:
        return "Nervous_Sense_Organs"
    if 390 <= code_num <= 459:
        return "Circulatory"
    if 460 <= code_num <= 519:
        return "Respiratory"
    if 520 <= code_num <= 579:
        return "Digestive"
    if 580 <= code_num <= 629:
        return "Genitourinary"
    if 630 <= code_num <= 679:
        return "Pregnancy_Childbirth"
    if 680 <= code_num <= 709:
        return "Skin"
    if 710 <= code_num <= 739:
        return "Musculoskeletal"
    if 740 <= code_num <= 759:
        return "Congenital"
    if 760 <= code_num <= 779:
        return "Perinatal"
    if 780 <= code_num <= 799:
        return "Symptoms"
    if 800 <= code_num <= 999:
        return "Injury_Poisoning"

    return "Other"


for diag_col in ["diag_1", "diag_2", "diag_3"]:
    if diag_col in df.columns:
        df[f"{diag_col}_group"] = df[diag_col].apply(map_icd9_group)

df[["diag_1", "diag_1_group", "diag_2", "diag_2_group", "diag_3", "diag_3_group"]].head()


## 8. Kiểm tra nhanh dữ liệu sau clean


In [ ]:
print("Final shape:", df.shape)
print("Duplicate rows:", df.duplicated().sum())

remaining_missing = (
    df.isna()
    .sum()
    .to_frame("missing_count")
    .assign(missing_rate=lambda x: x["missing_count"] / len(df))
    .query("missing_count > 0")
    .sort_values("missing_rate", ascending=False)
)

remaining_missing


In [ ]:
df.head()


## 9. Lưu dữ liệu clean chung

File này chưa phải là dữ liệu cuối cùng cho mô hình. Từ file này, ta sẽ tạo 3 pipeline riêng:

- Phân loại: encode, split train/test, scale nếu cần.
- Phân cụm: bỏ nhãn mục tiêu, encode, scale mạnh hơn.
- Luật kết hợp: rời rạc hóa biến số và chuyển sang transaction/basket one-hot.


In [ ]:
df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved common clean dataset to: {OUTPUT_PATH}")
